In [3]:
# The TF.js converter only supports Python <= 3.12, but Colab runs 3.13.
# So we give the converter its own small Python 3.12 environment (only used in Cell 7).
!pip install -q uv
!uv venv -q --python 3.12 /content/tfjs_env
!uv pip install -q --python /content/tfjs_env/bin/python tensorflowjs==4.22.0
!/content/tfjs_env/bin/tensorflowjs_converter --version


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 38.5 MB/s eta 0:00:00
2026-10-10 19:53:26.475105: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-10 19:53:26.482009: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-10 19:53:26.500602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791662006.534291    1753 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791662006.544079    1753 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791662006.568331    1753 computation_placer.cc:17

In [4]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # use the older Keras that the TF.js converter understands

import numpy as np
import tensorflow as tf
import tf_keras as keras
from google.colab import drive

drive.mount("/content/drive")
!unzip -q -o /content/drive/MyDrive/jordanwatch/dataset_v1.zip -d /content
print("TensorFlow", tf.__version__)

Mounted at /content/drive
TensorFlow 2.21.0


In [5]:
CLASSES = ["bloom", "vegetation", "clear_water", "not_water"]   # same order as Teachable Machine
SIZE = 224
BATCH = 32

# 85% of the train folder trains the model; 15% tells early stopping when to stop
train_ds = keras.utils.image_dataset_from_directory(
    "/content/dataset/train", class_names=CLASSES, label_mode="categorical",
    image_size=(SIZE, SIZE), batch_size=BATCH,
    validation_split=0.15, subset="training", seed=10)
val_ds = keras.utils.image_dataset_from_directory(
    "/content/dataset/train", class_names=CLASSES, label_mode="categorical",
    image_size=(SIZE, SIZE), batch_size=BATCH,
    validation_split=0.15, subset="validation", seed=10)
# the test folder is never used for training, only for the final score
test_ds = keras.utils.image_dataset_from_directory(
    "/content/dataset/test", class_names=CLASSES, label_mode="categorical",
    image_size=(SIZE, SIZE), batch_size=BATCH, shuffle=False)


def scale(images, labels):
    """Pixels 0-255 -> -1..1, exactly what the app will do to camera frames."""
    return images / 127.5 - 1.0, labels


def augment(images, labels):
    """Small random changes so the model sees more variety (training only)."""
    images = tf.image.random_flip_left_right(images)
    images = tf.image.random_brightness(images, 0.15)
    images = tf.image.random_contrast(images, 0.85, 1.15)
    return tf.clip_by_value(images, -1.0, 1.0), labels


train_ds = train_ds.map(scale).map(augment).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(scale).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(scale)

# class weights: rarer classes count more, and a missed bloom counts 1.5x extra
counts = np.array([len(os.listdir(f"/content/dataset/train/{c}")) for c in CLASSES])
weights = counts.sum() / (len(CLASSES) * counts)
weights[0] *= 1.5
class_weight = dict(enumerate(weights))
print(dict(zip(CLASSES, counts)))
print({c: round(w, 2) for c, w in zip(CLASSES, weights)})


Found 1469 files belonging to 4 classes.
Using 1249 files for training.
Found 1469 files belonging to 4 classes.
Using 220 files for validation.
Found 265 files belonging to 4 classes.
{'bloom': np.int64(383), 'vegetation': np.int64(401), 'clear_water': np.int64(434), 'not_water': np.int64(251)}
{'bloom': np.float64(1.44), 'vegetation': np.float64(0.92), 'clear_water': np.float64(0.85), 'not_water': np.float64(1.46)}


In [6]:
# MobileNetV2 already learned to see shapes, colors and textures from 1M+ photos.
# We freeze it and only train a small new last layer for our 4 classes (transfer learning).
base = keras.applications.MobileNetV2(
    input_shape=(SIZE, SIZE, 3), include_top=False, weights="imagenet", pooling="avg")
base.trainable = False

inputs = keras.Input(shape=(SIZE, SIZE, 3))
x = base(inputs, training=False)
x = keras.layers.Dropout(0.3)(x)                       # randomly hides 30% of features while training, against memorizing
outputs = keras.layers.Dense(len(CLASSES), activation="softmax")(x)   # 4 confidences that add up to 1
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


9406464/9406464 [==============================] - 2s 0us/step
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 mobilenetv2_1.00_224 (Func  (None, 1280)              2257984   
 tional)                                                         
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 4)                 5124      
                                                                 
Total params: 2263108 (8.63 MB)
Trainable params: 5124 (20.02 KB)
Non-trainable params: 2257984 (8.61 MB)
_________________________________________________________________


In [7]:
# stop when the validation loss hasn't improved for 3 epochs, and keep the best version
stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(train_ds, validation_data=val_ds, epochs=30,
                    class_weight=class_weight, callbacks=[stop])


Epoch 1/30
40/40 [==============================] - 10s 115ms/step - loss: 1.1927 - accuracy: 0.5733 - val_loss: 0.5483 - val_accuracy: 0.7955
Epoch 2/30
40/40 [==============================] - 5s 120ms/step - loss: 0.6257 - accuracy: 0.7806 - val_loss: 0.4550 - val_accuracy: 0.8318
Epoch 3/30
40/40 [==============================] - 4s 82ms/step - loss: 0.5134 - accuracy: 0.8255 - val_loss: 0.4273 - val_accuracy: 0.8500
Epoch 4/30
40/40 [==============================] - 3s 79ms/step - loss: 0.4563 - accuracy: 0.8535 - val_loss: 0.4354 - val_accuracy: 0.8455
Epoch 5/30
40/40 [==============================] - 5s 119ms/step - loss: 0.4318 - accuracy: 0.8527 - val_loss: 0.3917 - val_accuracy: 0.8545
Epoch 6/30
40/40 [==============================] - 4s 79ms/step - loss: 0.4114 - accuracy: 0.8519 - val_loss: 0.3932 - val_accuracy: 0.8591
Epoch 7/30
40/40 [==============================] - 4s 84ms/step - loss: 0.3378 - accuracy: 0.8967 - val_loss: 0.3785 - val_accuracy: 0.8455
Epoch 8/3

In [13]:
from sklearn.metrics import confusion_matrix, classification_report

probs = model.predict(test_ds)
truth = np.concatenate([labels.numpy() for _, labels in test_ds]).argmax(axis=1)
pred = probs.argmax(axis=1)
confidence = probs.max(axis=1)

print(f"accuracy: {(pred == truth).mean():.1%} on {len(truth)} test photos\n")
print("rows = real, columns = predicted:", CLASSES)
print(confusion_matrix(truth, pred), "\n")
print(classification_report(truth, pred, target_names=CLASSES))

for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    sure = confidence >= t
    wrong = sure & (pred != truth)
    missed_blooms = sure & (truth == 0) & (pred != 0)
    print(f"Uncertain below {t:.0%}: {(~sure).mean():.0%} uncertain, "
          f"{wrong.sum()} confident mistakes, {missed_blooms.sum()} confident missed blooms")


9/9 [==============================] - 1s 50ms/step
accuracy: 82.6% on 265 test photos

rows = real, columns = predicted: ['bloom', 'vegetation', 'clear_water', 'not_water']
[[58  3  9  2]
 [ 4 61  2  4]
 [ 4  3 66  4]
 [ 2  1  8 34]] 

              precision    recall  f1-score   support

       bloom       0.85      0.81      0.83        72
  vegetation       0.90      0.86      0.88        71
 clear_water       0.78      0.86      0.81        77
   not_water       0.77      0.76      0.76        45

    accuracy                           0.83       265
   macro avg       0.82      0.82      0.82       265
weighted avg       0.83      0.83      0.83       265

Uncertain below 50%: 6% uncertain, 38 confident mistakes, 11 confident missed blooms
Uncertain below 60%: 13% uncertain, 27 confident mistakes, 8 confident missed blooms
Uncertain below 70%: 17% uncertain, 22 confident mistakes, 8 confident missed blooms
Uncertain below 80%: 27% uncertain, 13 confident mistakes, 3 confident mi

In [9]:
model.save("/content/jordanwatch_v1.h5")
!/content/tfjs_env/bin/tensorflowjs_converter --input_format=keras /content/jordanwatch_v1.h5 /content/tfjs_model
!cd /content && zip -q -r jordanwatch-camera-v1-colab.zip tfjs_model

from google.colab import files
files.download("/content/jordanwatch-camera-v1-colab.zip")


/usr/local/lib/python3.13/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


2026-10-10 19:58:38.700062: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791662318.721155    3717 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791662318.727525    3717 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791662318.743171    3717 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791662318.743207    3717 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791662318.743211    3717 computation_placer.cc:177] computation placer alr

FileNotFoundError: Cannot find file: /content/jordanwatch-camera-v1-colab.zip

In [10]:
# The converter imports tensorflow_decision_forests (only for tree models, which we don't use),
# and that package is broken in this environment. Replace it with an empty stand-in.
SITE = "/content/tfjs_env/lib/python3.12/site-packages"
!rm -rf {SITE}/tensorflow_decision_forests
!mkdir -p {SITE}/tensorflow_decision_forests
!touch {SITE}/tensorflow_decision_forests/__init__.py

!/content/tfjs_env/bin/tensorflowjs_converter --input_format=keras /content/jordanwatch_v1.h5 /content/tfjs_model
!ls -la /content/tfjs_model
!cd /content && zip -q -r jordanwatch-camera-v1-colab.zip tfjs_model

from google.colab import files
files.download("/content/jordanwatch-camera-v1-colab.zip")


2026-10-10 20:00:08.076710: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791662408.115846    4095 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791662408.126136    4095 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791662408.150984    4095 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791662408.151020    4095 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791662408.151029    4095 computation_placer.cc:177] computation placer alr

FileNotFoundError: Cannot find file: /content/jordanwatch-camera-v1-colab.zip

In [11]:
# tensorflow_hub (imported by the converter) needs pkg_resources, which only older setuptools has
!uv pip install -q --python /content/tfjs_env/bin/python "setuptools<81"

!/content/tfjs_env/bin/tensorflowjs_converter --input_format=keras /content/jordanwatch_v1.h5 /content/tfjs_model
!ls -la /content/tfjs_model
!cd /content && zip -q -r jordanwatch-camera-v1-colab.zip tfjs_model

from google.colab import files
files.download("/content/jordanwatch-camera-v1-colab.zip")


2026-10-10 20:00:42.790774: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791662442.823797    4261 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791662442.833761    4261 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791662442.856570    4261 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791662442.856607    4261 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791662442.856616    4261 computation_placer.cc:177] computation placer alr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# Fine-tuning: let the top of MobileNetV2 adjust to our photos too, very gently.
base.trainable = True
for layer in base.layers[:-30]:      # everything except the last 30 layers stays frozen
    layer.trainable = False

# re-compile with a learning rate 100x smaller, so we nudge MobileNet instead of wrecking it
model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])

stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=20,
                       class_weight=class_weight, callbacks=[stop])


Epoch 1/20
40/40 [==============================] - 19s 118ms/step - loss: 0.2572 - accuracy: 0.9159 - val_loss: 0.3806 - val_accuracy: 0.8636
Epoch 2/20
40/40 [==============================] - 4s 96ms/step - loss: 0.2374 - accuracy: 0.9215 - val_loss: 0.3653 - val_accuracy: 0.8727
Epoch 3/20
40/40 [==============================] - 5s 101ms/step - loss: 0.1965 - accuracy: 0.9392 - val_loss: 0.3668 - val_accuracy: 0.8818
Epoch 4/20
40/40 [==============================] - 4s 95ms/step - loss: 0.2156 - accuracy: 0.9263 - val_loss: 0.3540 - val_accuracy: 0.8636
Epoch 5/20
40/40 [==============================] - 4s 98ms/step - loss: 0.1532 - accuracy: 0.9552 - val_loss: 0.3586 - val_accuracy: 0.8682
Epoch 6/20
40/40 [==============================] - 5s 118ms/step - loss: 0.1519 - accuracy: 0.9528 - val_loss: 0.3631 - val_accuracy: 0.8818
Epoch 7/20
40/40 [==============================] - 4s 88ms/step - loss: 0.1411 - accuracy: 0.9512 - val_loss: 0.3649 - val_accuracy: 0.8818
